# 5.2 - Few-Shot & In-Context Learning

**Module 5 - Prompt Engineering** | Netsetos GenAI Engineering

This notebook builds a support-ticket classifier that learns from examples alone - no training, no fine-tuning. You start with a single zero-shot-vs-few-shot A/B test, wrap the pattern in a reusable prompt builder, then add embedding-based retrieval so each query pulls its own most-relevant examples, and finish with one production-shaped class that ties it all together.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

Environment prep: bring in NumPy (used later for embedding vectors and cosine similarity) and pin the random seed so any seeded values in the lesson are reproducible run to run.

In [ ]:
# Install dependencies (if running in Colab or fresh environment)
# Uncomment the next line if needed:
# !pip install numpy -q

# Reproducibility - the lesson uses seeded random values throughout
import numpy as np
np.random.seed(42)

**Explanation**

A one-line environment cell, not a model call. It imports NumPy and fixes the random seed so results stay stable across reruns.

**How the code works - step by step**
- **`import numpy as np`** - NumPy backs every vector operation later (embeddings, dot products, norms).
- **`np.random.seed(42)`** - pins the RNG so any random draws are identical each run.
- The commented `pip install numpy -q` is there for a fresh Colab/venv - uncomment if NumPy is missing.

**In one line:** load NumPy and lock the seed.

**What you'll see:** (no output - environment prep)

## 1 - The shared helpers: `ask()` and `embed()`

Every example in the notebook goes through two tiny functions: `ask()` for a text completion and `embed()` for a vector. Both run on Google's current unified **google-genai** SDK (the older `google.generativeai` package was deprecated in 2025). Get these right once and the rest of the lesson is just calling them.

> **Needs a Gemini API key** (set `GOOGLE_API_KEY`).

In [ ]:
# pip install "google-genai>=1.0.0" numpy
from google import genai
from google.genai import types
import os, numpy as np

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

def ask(prompt: str, temperature: float = 0.1) -> str:
    """One text completion. Low temp for stable classification."""
    try:
        resp = client.models.generate_content(
            model="gemini-3-flash", contents=prompt,
            config=types.GenerateContentConfig(temperature=temperature))
        return resp.text.strip()
    except Exception as e:
        return f"[API error: {e}]"

def embed(text: str) -> np.ndarray:
    """One embedding vector for a piece of text."""
    r = client.models.embed_content(model="gemini-embedding-001", contents=text)
    return np.array(r.embeddings[0].values)

print(ask("Reply with one word: hello"))
# Output: Hello

**Explanation**

This is the plumbing cell - it builds one reusable client and two wrappers the whole lesson leans on. `ask()` sends a prompt and returns clean text at low temperature; `embed()` turns text into a NumPy vector you can compare by cosine similarity. Read it as: one client, one text call, one embedding call, defined once and reused everywhere.

**How the code works - step by step**
- **`genai.Client(api_key=...)`** - a single client object, created once and shared. The old `configure()` + per-call `GenerativeModel` pattern is gone.
- **`ask(prompt, temperature=0.1)`** - one completion on `gemini-3-flash`; the low default temperature keeps classification stable and repeatable. Its `try/except` returns an `[API error: ...]` string so one network blip can't crash a batch loop.
- **`embed(text)`** - one vector from `gemini-embedding-001`, returned as a NumPy array so you can take dot products and norms.
- **`print(ask(...))`** - a smoke test that the key and client actually work.

**In one line:** one client -> a stable text call and an embedding call, reused all lesson.

**What you'll see:** Prints a one-word smoke-test reply, `Hello`, confirming the API key and client are wired up correctly.

## 2 - Zero-shot vs few-shot: examples teach the pattern

Does showing examples actually help, or is it folklore? This cell proves it with a single controlled A/B: the exact same ticket, once with no examples and once with three. The examples don't add knowledge - they pin down *your* categories and *your* output format.

> **Needs a Gemini API key** (set `GOOGLE_API_KEY`).

In [ ]:
test = "My laptop won't connect to the company VPN since the update."

zero_shot = f"""Classify into exactly one of: Hardware, Software, Connectivity, Billing, Account.
Reply with ONLY the category.
Email: "{test}"
Category:"""

few_shot_prompt = f"""Classify support emails into exactly one category.

Email: "My monitor keeps flickering and showing weird colors."
Category: Hardware
Email: "I was charged twice for my subscription this month."
Category: Billing
Email: "WiFi keeps dropping every few minutes on my laptop."
Category: Connectivity

Email: "{test}"
Category:"""

print("zero-shot:", ask(zero_shot))
print("few-shot: ", ask(few_shot_prompt))
# Output: zero-shot: Software        (a reasonable guess, sometimes "Network")
# Output: few-shot:  Connectivity    (consistent, and always from your label set)

**Explanation**

A side-by-side experiment, not new machinery - it reuses `ask()` from the previous cell. Two prompts classify the identical VPN-connectivity email: one bare instruction, one carrying three labeled demonstrations. The point is to watch the answer move.

**How the code works - step by step**
- **`test`** - one ambiguous ticket about a laptop failing to reach the VPN after an update.
- **`zero_shot`** - lists the five categories and asks for one, with no examples; the model guesses your taxonomy.
- **`few_shot_prompt`** - prepends three `Email:`/`Category:` demonstrations that show the exact label set and format before the same test email.
- **Two `print(ask(...))` calls** - run both prompts so you can compare the answers directly.

**In one line:** same email, no examples vs three examples -> watch the label snap onto your set.

**What you'll see:** Prints `zero-shot: Software` (a plausible but off-list guess, sometimes "Network") and `few-shot: Connectivity` (consistent, and always drawn from your five labels).

## 3 - A reusable few-shot builder: format is the pattern

Rather than hand-writing prompts, wrap the pattern in a function. The single biggest lever in few-shot quality is a uniform template - every example in the exact same `Label: value` shape - so encode that once and never re-type it. The same builder then classifies sentiment, detects language, or extracts fields.

> **Needs a Gemini API key** (set `GOOGLE_API_KEY`).

In [ ]:
def few_shot(task: str, examples: list[tuple[str, str]],
             query: str, in_label="Input", out_label="Output") -> str:
    """Build a clean, uniform few-shot prompt from (input, output) pairs."""
    p = task.strip() + "\n\n"
    for inp, out in examples:                 # every example, identical shape
        p += f"{in_label}: {inp}\n{out_label}: {out}\n\n"
    p += f"{in_label}: {query}\n{out_label}:"     # query, no answer - model fills it
    return p

sentiment = [
    ("This product is amazing, best purchase ever.", "positive"),
    ("Terrible quality, broke in two days.", "negative"),
    ("It is okay, nothing special.", "neutral"),
    ("Love the design but the battery dies fast.", "mixed"),
]
prompt = few_shot(
    "Classify review sentiment. Reply with one word: positive, negative, neutral, or mixed.",
    sentiment, "Great camera, but support ignored me for a week.",
    "Review", "Sentiment")
print(ask(prompt))
# Output: mixed   (positive about the camera, negative about support)

**Explanation**

This cell factors the repetitive prompt-assembly into one `few_shot()` function, then exercises it on a four-way sentiment task. The function's whole job is to stamp every (input, output) pair into an identical shape and end on an open label so the model fills in exactly the next value. Read it as: a prompt template you configure with a task string, example pairs, and label names.

**How the code works - step by step**
- **`few_shot(task, examples, query, in_label, out_label)`** - starts with the task line, loops the example pairs into identical `{in_label}: ...\n{out_label}: ...` blocks, then appends the query and a trailing bare `{out_label}:` as the cue to answer.
- **`sentiment`** - four labeled reviews covering positive, negative, neutral, and mixed - one per class, so the full label space is shown.
- **The `few_shot(...)` call** - swaps the generic `Input`/`Output` labels for `Review`/`Sentiment` and passes an ambiguous camera-vs-support review as the query.
- **`print(ask(prompt))`** - classifies it.

**In one line:** one template, uniform separators, trailing open label -> reuse for any task.

**What you'll see:** Prints `mixed` - the model reads the ambiguous review (positive about the camera, negative about support) correctly, and the answer comes from the four-word label set the examples established.

## 4 - Dynamic example selection with embeddings

Stop using the same examples for every query. For each incoming ticket, retrieve the examples most *similar* to it from a labeled pool - the way you'd study the practice problems closest to the exam question. This is where the embeddings idea from Module 4 earns its keep.

> **Needs a Gemini API key** (set `GOOGLE_API_KEY`).

In [ ]:
# A labelled pool. Embed it ONCE and cache the vectors.
pool = [
    ("My phone screen is cracked and unresponsive", "Hardware"),
    ("The app crashes when I upload a photo", "Software"),
    ("WiFi keeps dropping on my laptop", "Connectivity"),
    ("I was double-charged for my subscription", "Billing"),
    ("How do I reset my account password?", "Account"),
    ("My promo code is rejected at checkout", "Billing"),
]
pool_vecs = [(text, label, embed(text)) for text, label in pool]

def nearest(query: str, k: int = 3):
    """Return the k examples whose vectors are closest to the query."""
    q = embed(query)
    scored = []
    for text, label, v in pool_vecs:
        cos = np.dot(q, v) / (np.linalg.norm(q) * np.linalg.norm(v))
        scored.append((cos, text, label))
    scored.sort(reverse=True)
    return [(t, l) for _, t, l in scored[:k]]

query = "There is an unauthorized charge of 49.99 on my account"
picked = nearest(query, k=3)
print("selected:", [l for _, l in picked])
prompt = few_shot("Classify the support ticket. Reply with one category.",
                  picked, query, "Ticket", "Category")
print("answer:  ", ask(prompt))
# Output: selected: ['Billing', 'Billing', 'Account']
# Output: answer:   Billing

**Explanation**

This cell adds retrieval on top of the builder. It embeds a small labeled pool once, then for a given query embeds only that query and ranks the pool by cosine similarity, feeding the top-k into `few_shot()`. Read it as: cache the pool vectors, measure angle-in-meaning-space per query, hand the nearest examples to the builder.

**How the code works - step by step**
- **`pool`** - six labeled tickets spanning all five categories.
- **`pool_vecs`** - embeds every pool entry once and caches `(text, label, vector)` so only the query is embedded per request.
- **`nearest(query, k=3)`** - embeds the query, computes cosine similarity (`dot / (norm * norm)`) against each cached vector, sorts descending, and returns the top-k (text, label) pairs.
- **The query + `few_shot(...)` + `ask(...)`** - a charge-dispute ticket retrieves its nearest examples, which flow straight into the builder from cell 3, then get classified.

**In one line:** embed the pool once -> rank by cosine per query -> feed the nearest into the builder.

**What you'll see:** Prints `selected: ['Billing', 'Billing', 'Account']` (the charge query pulled billing-heavy examples) and `answer: Billing` - correctly classified where a fixed Hardware/Software set would have guessed.

## 5 - Putting it together: a dynamic few-shot classifier

Collapse the whole lesson into one production-shaped class: embed a pool once, retrieve the nearest examples per query, build a uniform prompt, classify, and measure. This is the actual pattern behind real support-ticket routing and content moderation.

> **Needs a Gemini API key** (set `GOOGLE_API_KEY`).

In [ ]:
class DynamicFewShot:
    """Embed a pool once; retrieve the nearest examples per query."""
    def __init__(self, task: str, pool: list[tuple[str, str]]):
        self.task = task
        self.pool = [(t, l, embed(t)) for t, l in pool]   # cache vectors once

    def _nearest(self, query: str, k: int):
        q = embed(query)
        scored = [(float(np.dot(q, v) / (np.linalg.norm(q) * np.linalg.norm(v))), t, l)
                  for t, l, v in self.pool]
        scored.sort(reverse=True)
        return [(t, l) for _, t, l in scored[:k]]

    def classify(self, query: str, k: int = 3) -> str:
        prompt = few_shot(self.task, self._nearest(query, k), query, "Ticket", "Category")
        return ask(prompt)

clf = DynamicFewShot(
    "Classify the support ticket. Reply with one of: Hardware, Software, Connectivity, Billing, Account.",
    pool)   # the pool from Step 5

tests = [
    ("My keyboard makes a clicking sound but does not type", "Hardware"),
    ("I was charged for a plan I never signed up for", "Billing"),
    ("4G data stopped working after I traveled abroad", "Connectivity"),
]
correct = sum(clf.classify(q).lower().startswith(exp.lower()[:4]) for q, exp in tests)
print(f"dynamic: {correct}/{len(tests)} correct")
# Output: dynamic: 3/3 correct   (each query pulled its own relevant examples)

**Explanation**

The capstone cell - it packages cells 3 and 4 into a single reusable object and then scores it. `DynamicFewShot` caches its pool vectors at construction and exposes one `classify()` entry point; the test harness runs three unseen tickets and counts how many land correctly. Read it as: builder + embedded pool + retrieval, wrapped as a class, then measured against a held-out set.

**How the code works - step by step**
- **`__init__`** - stores the task string and embeds the pool once into `(text, label, vector)` triples, so vectors are computed a single time per classifier.
- **`_nearest`** - the same cosine-similarity ranking as cell 4, returning the k closest (text, label) pairs.
- **`classify`** - calls `_nearest`, feeds the result into `few_shot()`, and returns `ask()`'s answer - one line per query.
- **The test block** - builds a classifier over the pool, runs three fresh tickets, and counts a hit when the answer's first four letters match the expected label (case-insensitive).

**In one line:** embed pool once -> retrieve per query -> build -> classify -> score.

**What you'll see:** Prints `dynamic: 3/3 correct` - each of the three test tickets pulled its own relevant examples and was classified correctly.

Across six cells you went from "does showing examples even help?" to a measured, production-shaped classifier. The through-line: examples specify the *task* (its format and label space), not new facts - so a uniform template matters more than any single example, and retrieving the nearest examples per query beats one fixed set. The final `DynamicFewShot` class is the pattern real support-ticket routing and content moderation run on. Next, Lesson 5.3 covers when to stop showing examples and let a reasoning model think, and Lesson 5.6 automates the example-selection step you built by hand here.